# Paper 3 — Notebook 03: Baselines & the Optimism Gap

**This notebook produces the headline result of the paper.**

### The question
When people evaluate machine-learning models on melt-pool data collected from many published
studies, they usually shuffle all the rows randomly and hold out 10% for testing. But rows from the
*same source study* then appear in both training and testing — same lab, same machine, same setup.
The model has effectively seen the answer key.

A more realistic test holds out **entire source studies**: train on some studies, test on studies the
model has never seen. That mimics real deployment, where a new lab or new machine produces data
the model was never trained on.

### The three protocols compared here
| Protocol | What is held out | What it simulates |
|---|---|---|
| **V0** | random rows | the optimistic (standard) practice |
| **V1** | entire source studies (`paper ID`) | a new lab / unseen study |
| **V2** | one entire alloy at a time | a brand-new material |

### What we measure
The **optimism gap** = how much worse V1 (and V2) is than V0. A large gap means the standard
practice overstates real-world performance.

### Guardrails used throughout
- **Nested tuning inside groups.** Hyper-parameters are selected using an inner split of the
  *training* studies only. The held-out study is never used for tuning — otherwise the "unseen
  study" claim is contaminated.
- **Pooled out-of-fold (OOF) predictions.** Every row is predicted exactly once, while it was held
  out. Metrics are computed on the pooled predictions, not averaged per fold. This is important for
  the rare `balling` class, which is thin in some grouped folds.
- **Imputation happens inside the pipeline**, so it is re-fitted on each training split and never
  leaks information from the test split.

> **Runtime:** roughly **20–30 minutes** end-to-end on a free Colab CPU instance (no GPU needed).
> Sections 4, 5 and 6 are the slow parts. All results are written to `./results/` as CSVs so later
> notebooks and the manuscript read from saved files rather than re-running anything.


## 0 · Setup and load frozen data

In [ ]:
import importlib.util, subprocess, sys
PKGS = {'pandas':'pandas','numpy':'numpy','scikit-learn':'sklearn','pyarrow':'pyarrow',
        'xgboost':'xgboost','matplotlib':'matplotlib'}
missing=[p for p,i in PKGS.items() if importlib.util.find_spec(i) is None]
if missing:
    cmd=[sys.executable,'-m','pip','install','-q',*missing]
    if subprocess.run(cmd).returncode!=0: subprocess.run(cmd+['--break-system-packages'])
print('environment ready')

In [ ]:
PREP_CODE = r"""
import os, re, json, hashlib, subprocess
import numpy as np, pandas as pd
from sklearn.model_selection import (KFold, GroupKFold, StratifiedKFold, StratifiedGroupKFold)

SEED = 42
np.random.seed(SEED)
ART = 'artifacts'
os.makedirs(ART, exist_ok=True)

# clone data if not present
if not os.path.exists('MeltpoolNet/Data/meltpoolnet_regression.csv'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/BaratiLab/MeltpoolNet.git'], check=True)

REG_PATH = 'MeltpoolNet/Data/meltpoolnet_regression.csv'
CLS_PATH = 'MeltpoolNet/Data/meltpoolnet_classification.csv'
print('seed', SEED, '| artifacts ->', os.path.abspath(ART))

reg = pd.read_csv(REG_PATH)
cls = pd.read_csv(CLS_PATH)
print('regression   :', reg.shape)
print('classification:', cls.shape)

def drop_junk(df):
    junk = [c for c in df.columns if c.startswith('Unnamed:') or str(c).strip()=='']
    for extra in ['comment']:
        if extra in df.columns: junk.append(extra)
    return df.drop(columns=junk), junk

reg, jr = drop_junk(reg)
cls, jc = drop_junk(cls)
print('dropped from regression   :', jr)
print('dropped from classification:', jc)

CLASSES = ['desirable','keyhole','LOF','balling']   # fixed id order: 0,1,2,3
CLS_ID = {c:i for i,c in enumerate(CLASSES)}

print('classification labels BEFORE:', dict(cls['meltpool shape'].value_counts(dropna=False)))
cls = cls[cls['meltpool shape'].isin(CLASSES)].copy()
cls['y_class'] = cls['meltpool shape'].map(CLS_ID).astype(int)
print('classification labels AFTER :', dict(cls['meltpool shape'].value_counts()))

# The regression file ALSO carries meltpool shape -> used for the multi-task class head.
reg_class_mask = reg['meltpool shape'].isin(CLASSES)
print('\nregression-file class labels (multi-task head):',
      dict(reg.loc[reg_class_mask,'meltpool shape'].value_counts()))

def comp_cols(df):
    return [c for c in df.columns if re.search(r'\(wt\.?%\)', c)]

# --- regression feature ladder ---
reg_F1 = ['Power','Velocity','powder flowrate','layer thickness','beam D','Hatch spacing']
reg_F2 = reg_F1 + ['density','Cp','k','melting T','absorption coefficient','minimum absorptivity']
reg_F3 = reg_F2 + ['E (J/mm)','E (J/mm3)']
reg_F4 = reg_F3 + comp_cols(reg)
reg_F5 = reg_F3 + ['Material']            # diagnostic only

# --- classification feature ladder ---
cls_F1 = ['Power','Velocity','Hatch spacing','layer thickness','beam D']
cls_F2 = cls_F1 + ['density','Cp','k','melting T','absorption coefficient','minimal absorptivity']
cls_F3 = cls_F2 + ['p/lb','p/l','p/b2','p/b','vb','vl']
cls_F4 = cls_F3 + comp_cols(cls)
cls_F5 = cls_F3 + ['Material']            # diagnostic only

REG_FEATURES = {'F1':reg_F1,'F2':reg_F2,'F3':reg_F3,'F4':reg_F4,'F5':reg_F5}
CLS_FEATURES = {'F1':cls_F1,'F2':cls_F2,'F3':cls_F3,'F4':cls_F4,'F5':cls_F5}

# verify every listed column exists
for tag, fd, df in [('reg',REG_FEATURES,reg),('cls',CLS_FEATURES,cls)]:
    for name, cols in fd.items():
        miss = [c for c in cols if c not in df.columns]
        assert not miss, f'{tag} {name} missing {miss}'
        print(f'{tag} {name}: {len(cols)} cols  (missing: {miss})')

# leakage assertion
LEAKY = {'d/l','d/w','l/w','depth of meltpool','width of melt pool','length of melt pool',
         'spatter','porosity','relative density','meltpool shape','paper ID','paper'}
for fd in (REG_FEATURES, CLS_FEATURES):
    for cols in fd.values():
        assert not (set(cols) & LEAKY), f'LEAK: {set(cols)&LEAKY}'
print('\nleakage guard passed: no target-derived or outcome columns in any feature set')

def coerce_numeric(df, cols):
    for c in cols:
        if c != 'Material':
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

reg = coerce_numeric(reg, sorted(set(sum(REG_FEATURES.values(), []))))
cls = coerce_numeric(cls, sorted(set(sum(CLS_FEATURES.values(), []))))

reg = reg.reset_index(drop=True)
cls = cls.reset_index(drop=True)

# missingness snapshot for the richest feature set (F4)
print('regression F4 missingness (top 8):')
print(reg[reg_F4].isna().mean().sort_values(ascending=False).head(8).round(2).to_string())
print('\nNaNs are RETAINED — downstream models must be native-NaN (XGBoost) or impute inside a')
print('fold-fitted pipeline, never globally.')

reg['has_depth'] = reg['depth of meltpool'].notna()
reg['has_width'] = reg['width of melt pool'].notna()
reg['has_class'] = reg['meltpool shape'].isin(CLASSES)
reg['y_class']   = reg['meltpool shape'].map(CLS_ID)   # NaN where no class label

print(f"has_depth = {reg['has_depth'].sum()}")
print(f"has_width = {reg['has_width'].sum()}")
print(f"has_class = {reg['has_class'].sum()}")
print(f"depth&width (complete-case for single-vs-multi comparison) = "
      f"{(reg['has_depth']&reg['has_width']).sum()}")

K_REG, K_CLS = 10, 5

def reg_splits(mask_col, k=K_REG):
    m = reg[mask_col].values
    idx = np.where(m)[0]
    groups = reg.loc[idx, 'paper ID'].values
    v0 = np.full(len(reg), -1); v1 = np.full(len(reg), -1)
    for f,(_,te) in enumerate(KFold(k, shuffle=True, random_state=SEED).split(idx)):
        v0[idx[te]] = f
    for f,(_,te) in enumerate(GroupKFold(n_splits=k).split(idx, groups=groups)):
        v1[idx[te]] = f
    return v0, v1

reg['v0_depth'], reg['v1_depth'] = reg_splits('has_depth')
reg['v0_width'], reg['v1_width'] = reg_splits('has_width')

def lomo_materials(mask_col, min_rows=40):
    vc = reg.loc[reg[mask_col], 'Material'].value_counts()
    return vc[vc >= min_rows].index.tolist()

LOMO_DEPTH = lomo_materials('has_depth')
LOMO_WIDTH = lomo_materials('has_width')
print('depth fold sizes  V0:', [int((reg['v0_depth']==f).sum()) for f in range(K_REG)])
print('depth fold sizes  V1:', [int((reg['v1_depth']==f).sum()) for f in range(K_REG)])
print('LOMO depth materials:', LOMO_DEPTH)
print('LOMO width materials:', LOMO_WIDTH)

# classification splits (stratified; grouped variant respects paper ID)
cy, cg = cls['y_class'].values, cls['paper ID'].values
cls['v0'] = -1; cls['v1'] = -1
for f,(_,te) in enumerate(StratifiedKFold(K_CLS, shuffle=True, random_state=SEED).split(cls, cy)):
    cls.loc[te,'v0'] = f
for f,(_,te) in enumerate(StratifiedGroupKFold(K_CLS, shuffle=True, random_state=SEED).split(cls, cy, groups=cg)):
    cls.loc[te,'v1'] = f
LOMO_CLS = cls['Material'].value_counts()[lambda s: s>=40].index.tolist()
print('\nclassification LOMO materials:', LOMO_CLS)

# 8a. no source-group appears in both train and test of any grouped fold
def assert_group_disjoint(df, fold_col, group_col, k):
    for f in range(k):
        te = set(df.loc[df[fold_col]==f, group_col])
        tr = set(df.loc[(df[fold_col]!=f) & (df[fold_col]!=-1), group_col])
        assert te.isdisjoint(tr), f'{fold_col}: group leak in fold {f}'
assert_group_disjoint(reg[reg['has_depth']], 'v1_depth', 'paper ID', K_REG)
assert_group_disjoint(reg[reg['has_width']], 'v1_width', 'paper ID', K_REG)
assert_group_disjoint(cls, 'v1', 'paper ID', K_CLS)

# 8b. every labelled row is assigned to exactly one eval fold
assert (reg.loc[reg['has_depth'],'v1_depth']>=0).all()
assert (reg.loc[reg['has_width'],'v1_width']>=0).all()
assert (cls['v1']>=0).all()

# 8c. balling-per-fold warning (data-driven, not a failure) — see notebook 03 for pooled OOF metric
print('classification V1 balling count per fold (thin folds are expected & handled via pooled OOF):')
for f in range(K_CLS):
    print(f'  fold {f}: balling =', int(((cls["v1"]==f)&(cls["y_class"]==CLS_ID["balling"])).sum()))
print('\nALL SANITY CHECKS PASSED')

# cleaned frames (parquet keeps dtypes + NaNs)
reg.to_parquet(f'{ART}/reg_clean.parquet', index=False)
cls.to_parquet(f'{ART}/cls_clean.parquet', index=False)

feature_sets = {'regression': REG_FEATURES, 'classification': CLS_FEATURES}
with open(f'{ART}/feature_sets.json','w') as f:
    json.dump(feature_sets, f, indent=2)

meta = {
    'seed': SEED,
    'source_key': 'paper ID',
    'classes': CLASSES,
    'regression': {
        'primary_target': 'depth of meltpool',
        'secondary_target': 'width of melt pool',
        'appendix_target': 'length of melt pool',
        'k_folds': K_REG,
        'n_depth': int(reg['has_depth'].sum()),
        'n_width': int(reg['has_width'].sum()),
        'n_class_labels': int(reg['has_class'].sum()),
        'n_depth_and_width': int((reg['has_depth']&reg['has_width']).sum()),
        'lomo_depth_materials': LOMO_DEPTH,
        'lomo_width_materials': LOMO_WIDTH,
    },
    'classification': {
        'k_folds': K_CLS,
        'class_counts': {c:int((cls['y_class']==i).sum()) for c,i in CLS_ID.items()},
        'lomo_materials': LOMO_CLS,
        'note': 'grouped folds strand balling in some folds; use pooled out-of-fold predictions '
                'for macro-F1, not per-fold averaging.',
    },
    'data_commit': subprocess.run(['git','-C','MeltpoolNet','rev-parse','HEAD'],
                                  capture_output=True, text=True).stdout.strip(),
}
with open(f'{ART}/meta.json','w') as f:
    json.dump(meta, f, indent=2)

print('written:')
for p in ['reg_clean.parquet','cls_clean.parquet','feature_sets.json','meta.json']:
    print('  ', f'{ART}/{p}')
print('\nmeta.json:')
print(json.dumps(meta, indent=2))
"""
print("preparation code loaded (", len(PREP_CODE), "chars )")

## 0b · Auto-prepare data (self-contained)

If `artifacts/` is missing — for example Colab restarted, or Notebook 02 was run in a different
session — this cell rebuilds it automatically by running the Notebook 02 logic. If the artifacts are
already present it does nothing. **You never have to run Notebook 02 separately.**

In [ ]:
import os
NEED = ['artifacts/reg_clean.parquet','artifacts/cls_clean.parquet',
        'artifacts/feature_sets.json','artifacts/meta.json']
if all(os.path.exists(p) for p in NEED):
    print('artifacts already present — skipping preparation')
else:
    print('artifacts missing -> running Notebook 02 preparation now...\n')
    exec(PREP_CODE)
    print('\npreparation complete')


In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR, SVC
from sklearn.model_selection import GroupKFold, KFold, GridSearchCV
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             f1_score, balanced_accuracy_score, matthews_corrcoef,
                             classification_report, confusion_matrix)
from xgboost import XGBRegressor, XGBClassifier

SEED = 42
ART, RES = 'artifacts', 'results'
os.makedirs(RES, exist_ok=True)

assert os.path.exists(f'{ART}/reg_clean.parquet'), 'Run Notebook 02 first.'
reg  = pd.read_parquet(f'{ART}/reg_clean.parquet')
cls  = pd.read_parquet(f'{ART}/cls_clean.parquet')
FS   = json.load(open(f'{ART}/feature_sets.json'))
META = json.load(open(f'{ART}/meta.json'))
CLASSES = META['classes']
print('regression rows:', len(reg), '| classification rows:', len(cls))
print('depth labels:', int(reg['has_depth'].sum()), '| width labels:', int(reg['has_width'].sum()))

## 1 · Model zoo

Deliberately standard models. The paper's contribution is the *evaluation*, not a new architecture —
so the baselines must be strong, familiar, and fairly tuned.

`XGBoost` handles missing values natively, so it gets no imputation step. The others get median
imputation *inside* the pipeline.

In [ ]:
def reg_model(name):
    if name=='Ridge':
        return (Pipeline([('imp',SimpleImputer(strategy='median')),
                          ('sc',StandardScaler()),('m',Ridge())]),
                {'m__alpha':[0.1,1.0,10.0,100.0]})
    if name=='RF':
        return (Pipeline([('imp',SimpleImputer(strategy='median')),
                          ('m',RandomForestRegressor(random_state=SEED,n_jobs=-1))]),
                {'m__n_estimators':[300],'m__max_depth':[None,10],'m__min_samples_leaf':[1,3]})
    if name=='SVR':
        return (Pipeline([('imp',SimpleImputer(strategy='median')),
                          ('sc',StandardScaler()),('m',SVR())]),
                {'m__C':[1,10,100],'m__gamma':['scale'],'m__epsilon':[0.1]})
    if name=='XGB':
        return (Pipeline([('m',XGBRegressor(random_state=SEED,n_estimators=400,
                                            verbosity=0,n_jobs=-1))]),
                {'m__max_depth':[3,6],'m__learning_rate':[0.05,0.1],'m__subsample':[0.8]})
    raise ValueError(name)

def clf_model(name):
    if name=='LogReg':
        return (Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),
                          ('m',LogisticRegression(max_iter=2000,class_weight='balanced'))]),
                {'m__C':[0.1,1.0,10.0]})
    if name=='RF':
        return (Pipeline([('imp',SimpleImputer(strategy='median')),
                          ('m',RandomForestClassifier(random_state=SEED,n_jobs=-1,
                                                      class_weight='balanced'))]),
                {'m__n_estimators':[300],'m__max_depth':[None,10],'m__min_samples_leaf':[1,3]})
    if name=='SVC':
        return (Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),
                          ('m',SVC(class_weight='balanced'))]),
                {'m__C':[1,10,100],'m__gamma':['scale']})
    if name=='XGB':
        return (Pipeline([('m',XGBClassifier(random_state=SEED,n_estimators=400,verbosity=0,
                                             n_jobs=-1,objective='multi:softprob',
                                             num_class=len(CLASSES)))]),
                {'m__max_depth':[3,6],'m__learning_rate':[0.05,0.1],'m__subsample':[0.8]})
    raise ValueError(name)

REG_MODELS = ['Ridge','RF','SVR','XGB']
CLF_MODELS = ['LogReg','RF','SVC','XGB']
print('regression models:', REG_MODELS)
print('classification models:', CLF_MODELS)

## 2 · The evaluation engine

One function produces pooled out-of-fold predictions for any (data, protocol, model) combination.
The only difference between protocols is **which rows are held out together** and **how the inner
tuning split is formed**.

In [ ]:
def oof_predict(X, y, fold_ids, groups, build_model, model_name,
                task='reg', inner_k=3, proba=False):
    """Return pooled out-of-fold predictions. Each row predicted once, while held out."""
    n = len(y)
    oof = np.full(n, np.nan)
    oof_p = np.full((n, len(CLASSES)), np.nan) if proba else None
    folds = sorted(f for f in np.unique(fold_ids) if f >= 0)

    for f in folds:
        te = fold_ids == f
        tr = (fold_ids != f) & (fold_ids >= 0)
        if te.sum() == 0 or tr.sum() == 0:
            continue
        est, grid = build_model(model_name)
        scoring = 'neg_mean_absolute_error' if task=='reg' else 'f1_macro'

        # --- nested tuning: grouped inner split when groups are meaningful ---
        g_tr = groups[tr] if groups is not None else None
        use_grouped_inner = g_tr is not None and len(np.unique(g_tr)) >= inner_k
        if use_grouped_inner:
            gs = GridSearchCV(est, grid, cv=GroupKFold(n_splits=inner_k),
                              scoring=scoring, n_jobs=-1)
            gs.fit(X[tr], y[tr], groups=g_tr)
        else:
            gs = GridSearchCV(est, grid, cv=KFold(inner_k, shuffle=True, random_state=SEED),
                              scoring=scoring, n_jobs=-1)
            gs.fit(X[tr], y[tr])

        best = gs.best_estimator_
        oof[te] = best.predict(X[te])
        if proba and hasattr(best, 'predict_proba'):
            oof_p[te] = best.predict_proba(X[te])
    return (oof, oof_p) if proba else oof


def lomo_folds(frame, mask, materials):
    """Leave-one-material-out fold ids: fold i = rows of materials[i]; others -1."""
    ids = np.full(len(frame), -1)
    for i, m in enumerate(materials):
        ids[(frame['Material'].values == m) & mask] = i
    return ids
print('engine ready')

## 3 · Metrics with group-level bootstrap confidence intervals

Resampling is done **by source study**, not by row. Rows within a study are correlated, so
row-level bootstrapping would understate the uncertainty.

In [ ]:
def boot_ci_reg(y, yhat, groups, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    uniq = np.unique(groups)
    maes, r2s = [], []
    for _ in range(n_boot):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([np.where(groups==g)[0] for g in pick])
        if len(idx) < 5: continue
        maes.append(mean_absolute_error(y[idx], yhat[idx]))
        r2s.append(r2_score(y[idx], yhat[idx]))
    q = lambda a: (float(np.percentile(a,2.5)), float(np.percentile(a,97.5)))
    return q(maes), q(r2s)

def reg_metrics(y, yhat, groups=None, label=''):
    m = {'label':label,
         'MAE': float(mean_absolute_error(y,yhat)),
         'RMSE': float(np.sqrt(mean_squared_error(y,yhat))),
         'R2': float(r2_score(y,yhat)),
         'n': int(len(y))}
    if groups is not None:
        (lo,hi),(r_lo,r_hi) = boot_ci_reg(y,yhat,groups)
        m.update({'MAE_lo':lo,'MAE_hi':hi,'R2_lo':r_lo,'R2_hi':r_hi})
    return m

def clf_metrics(y, yhat, label=''):
    return {'label':label,
            'macroF1': float(f1_score(y,yhat,average='macro')),
            'balanced_acc': float(balanced_accuracy_score(y,yhat)),
            'MCC': float(matthews_corrcoef(y,yhat)),
            'accuracy': float((y==yhat).mean()),
            'n': int(len(y))}
print('metrics ready')

## 4 · EXPERIMENT A — Regression: depth (primary target)

Feature set **F3** (process + material thermophysical properties + energy density) is used for the
headline comparison. Runtime is roughly 3–5 minutes.

In [ ]:
FEATS = FS['regression']['F3']
TARGET = 'depth of meltpool'

sub    = reg[reg['has_depth']].reset_index(drop=True)
Xd     = sub[FEATS]
yd     = sub[TARGET].values
gd     = sub['paper ID'].values
lomo_d = lomo_folds(sub, np.ones(len(sub),bool), META['regression']['lomo_depth_materials'])

protocols = {
    'V0_random'   : (sub['v0_depth'].values, None),   # inner tuning: plain KFold
    'V1_by_study' : (sub['v1_depth'].values, gd),     # inner tuning: grouped
    'V2_by_alloy' : (lomo_d,                 gd),
}

rows_depth = []
t0=time.time()
for mname in REG_MODELS:
    for pname,(folds,grp) in protocols.items():
        t=time.time()
        oof = oof_predict(Xd, yd, folds, grp, reg_model, mname, task='reg')
        ok  = ~np.isnan(oof)
        m   = reg_metrics(yd[ok], oof[ok], gd[ok], label=f'{mname}|{pname}')
        m.update(model=mname, protocol=pname, target='depth', features='F3',
                 seconds=round(time.time()-t,1))
        rows_depth.append(m)
        print(f"{mname:6s} {pname:12s} MAE={m['MAE']:7.2f}  R2={m['R2']:6.3f}  ({m['seconds']}s)")
depth_df = pd.DataFrame(rows_depth)
depth_df.to_csv(f'{RES}/depth_baselines.csv', index=False)
print('\ntotal', round(time.time()-t0,1), 's  ->', f'{RES}/depth_baselines.csv')

### 4.1 · The optimism gap for depth

In [ ]:
piv = depth_df.pivot_table(index='model', columns='protocol', values=['MAE','R2'])
print(piv.round(3).to_string())

gap = []
for mname in REG_MODELS:
    r  = depth_df[depth_df.model==mname].set_index('protocol')
    for p in ['V1_by_study','V2_by_alloy']:
        gap.append({'model':mname,'protocol':p,
                    'MAE_V0':r.loc['V0_random','MAE'], 'MAE_p':r.loc[p,'MAE'],
                    'MAE_ratio':r.loc[p,'MAE']/r.loc['V0_random','MAE'],
                    'R2_V0':r.loc['V0_random','R2'], 'R2_p':r.loc[p,'R2'],
                    'R2_drop':r.loc['V0_random','R2']-r.loc[p,'R2']})
gap_df = pd.DataFrame(gap)
gap_df.to_csv(f'{RES}/depth_optimism_gap.csv', index=False)
print('\n--- OPTIMISM GAP (depth) ---')
print(gap_df.round(3).to_string(index=False))
print('\nMAE_ratio = how many times WORSE the honest protocol is than the random split.')

## 5 · EXPERIMENT B — Feature-set ladder under the honest protocol

Does adding **material physics** (F2) and **energy-density** features (F3) help the model transfer
to unseen studies and unseen alloys? `F5` adds the material name as a categorical and is a
**memorization diagnostic only** — a large V0 gain with no V1/V2 gain is evidence the model is
keying on material identity rather than physics.

In [ ]:
ladder = []
for fset in ['F1','F2','F3']:
    cols = FS['regression'][fset]
    Xl = sub[cols]
    for pname,(folds,grp) in protocols.items():
        oof = oof_predict(Xl, yd, folds, grp, reg_model, 'XGB', task='reg')
        ok  = ~np.isnan(oof)
        m = reg_metrics(yd[ok], oof[ok], gd[ok], label=f'XGB|{fset}|{pname}')
        m.update(model='XGB', protocol=pname, features=fset, target='depth')
        ladder.append(m)
        print(f"XGB {fset} {pname:12s} MAE={m['MAE']:7.2f}  R2={m['R2']:6.3f}")
ladder_df = pd.DataFrame(ladder)
ladder_df.to_csv(f'{RES}/depth_feature_ladder.csv', index=False)
print('\n', ladder_df.pivot_table(index='features',columns='protocol',
                                  values=['MAE','R2']).round(3).to_string())

### 5.1 · Which alloys are hardest to predict when unseen?

Under V2 each alloy is held out entirely. Per-alloy error shows *where* transfer fails, which is far
more informative for the paper than a single pooled number.

In [ ]:
oof_v2 = oof_predict(sub[FS['regression']['F3']], yd, lomo_d, gd, reg_model, 'XGB', task='reg')
ok = ~np.isnan(oof_v2)
per_alloy = []
for i, mat in enumerate(META['regression']['lomo_depth_materials']):
    sel = (lomo_d == i) & ok
    if sel.sum() > 5:
        per_alloy.append({'material':mat, 'n':int(sel.sum()),
                          'MAE':float(mean_absolute_error(yd[sel], oof_v2[sel])),
                          'mean_depth':float(np.mean(yd[sel])),
                          'MAE_pct_of_mean':float(100*mean_absolute_error(yd[sel],oof_v2[sel])/np.mean(yd[sel]))})
alloy_df = pd.DataFrame(per_alloy).sort_values('MAE')
alloy_df.to_csv(f'{RES}/depth_per_alloy_V2.csv', index=False)
print(alloy_df.round(2).to_string(index=False))
print('\nLarge spread here means transferability is alloy-dependent — a key discussion point.')

## 6 · EXPERIMENT C — Classification (4 defect modes)

Grouped folds sometimes leave very few `balling` samples in a fold, so metrics are computed on
**pooled** out-of-fold predictions across all folds — never averaged per fold.

In [ ]:
cfeats = FS['classification']['F3']
Xc = cls[cfeats]
yc = cls['y_class'].values
gc = cls['paper ID'].values
lomo_c = lomo_folds(cls, np.ones(len(cls),bool), META['classification']['lomo_materials'])

cprot = {'V0_random':(cls['v0'].values,None),
         'V1_by_study':(cls['v1'].values,gc),
         'V2_by_alloy':(lomo_c,gc)}

rows_cls=[]
for mname in CLF_MODELS:
    for pname,(folds,grp) in cprot.items():
        t=time.time()
        oof = oof_predict(Xc, yc, folds, grp, clf_model, mname, task='clf')
        ok  = ~np.isnan(oof)
        m = clf_metrics(yc[ok], oof[ok].astype(int), label=f'{mname}|{pname}')
        m.update(model=mname, protocol=pname, features='F3', seconds=round(time.time()-t,1))
        rows_cls.append(m)
        print(f"{mname:7s} {pname:12s} macroF1={m['macroF1']:.3f}  "
              f"balAcc={m['balanced_acc']:.3f}  MCC={m['MCC']:.3f}  ({m['seconds']}s)")
cls_df = pd.DataFrame(rows_cls)
cls_df.to_csv(f'{RES}/classification_baselines.csv', index=False)
print('\n', cls_df.pivot_table(index='model',columns='protocol',
                               values=['macroF1','balanced_acc','MCC']).round(3).to_string())

### 6.1 · Per-class breakdown for the best model (honest protocol)

In [ ]:
best = cls_df[cls_df.protocol=='V1_by_study'].sort_values('macroF1',ascending=False).iloc[0]['model']
print('best model under V1_by_study:', best, '\n')
oof_best = oof_predict(Xc, yc, cls['v1'].values, gc, clf_model, best, task='clf')
ok = ~np.isnan(oof_best)
print(classification_report(yc[ok], oof_best[ok].astype(int),
                            target_names=CLASSES, digits=3))
cm = confusion_matrix(yc[ok], oof_best[ok].astype(int))
print('confusion matrix (rows=true, cols=pred):')
print(pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_string())

## 7 · Headline figure — the optimism gap

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1,3, figsize=(15,4.2))
order = ['V0_random','V1_by_study','V2_by_alloy']
labels = ['V0\nrandom split','V1\nunseen study','V2\nunseen alloy']

# (a) depth MAE
ax=axes[0]
for mname in REG_MODELS:
    r = depth_df[depth_df.model==mname].set_index('protocol').loc[order]
    ax.plot(range(3), r['MAE'], marker='o', label=mname)
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_ylabel('MAE (µm)  — lower is better'); ax.set_title('(a) Melt-pool depth: error')
ax.legend(fontsize=8); ax.grid(alpha=.3)

# (b) depth R2  — clipped so one badly-transferring model doesn't squash the others
ax=axes[1]
for mname in REG_MODELS:
    r = depth_df[depth_df.model==mname].set_index('protocol').loc[order]
    ax.plot(range(3), r['R2'], marker='o', label=mname)
ax.axhline(0, color='k', lw=.8, ls='--')
lo = min(-0.5, depth_df['R2'].replace(-np.inf,np.nan).min())
ax.set_ylim(max(lo,-2.0), 1.05)   # values below -2 are annotated, not plotted to scale
off = depth_df[depth_df['R2'] < -2.0]
if len(off):
    ax.text(0.02, 0.04, 'note: '+', '.join(f"{r.model}/{r.protocol.split('_')[0]} R²={r.R2:.1f}"
            for r in off.itertuples()) + ' (below axis)',
            transform=ax.transAxes, fontsize=7, color='dimgray')
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_ylabel('R²  — higher is better'); ax.set_title('(b) Melt-pool depth: R²')
ax.grid(alpha=.3)

# (c) classification macro-F1
ax=axes[2]
for mname in CLF_MODELS:
    r = cls_df[cls_df.model==mname].set_index('protocol').loc[order]
    ax.plot(range(3), r['macroF1'], marker='s', label=mname)
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_ylabel('macro-F1  — higher is better'); ax.set_title('(c) Defect mode: macro-F1')
ax.legend(fontsize=8); ax.grid(alpha=.3)

plt.tight_layout()
plt.savefig(f'{RES}/fig_optimism_gap.png', dpi=200, bbox_inches='tight')
print('saved', f'{RES}/fig_optimism_gap.png')
plt.show()

## 8 · Summary — what to put in the paper

In [ ]:
summary = {
 'depth': {
   'best_model_V0': depth_df[depth_df.protocol=='V0_random'].sort_values('MAE').iloc[0][['model','MAE','R2']].to_dict(),
   'best_model_V1': depth_df[depth_df.protocol=='V1_by_study'].sort_values('MAE').iloc[0][['model','MAE','R2']].to_dict(),
   'best_model_V2': depth_df[depth_df.protocol=='V2_by_alloy'].sort_values('MAE').iloc[0][['model','MAE','R2']].to_dict(),
 },
 'classification': {
   'best_V0': cls_df[cls_df.protocol=='V0_random'].sort_values('macroF1',ascending=False).iloc[0][['model','macroF1']].to_dict(),
   'best_V1': cls_df[cls_df.protocol=='V1_by_study'].sort_values('macroF1',ascending=False).iloc[0][['model','macroF1']].to_dict(),
   'best_V2': cls_df[cls_df.protocol=='V2_by_alloy'].sort_values('macroF1',ascending=False).iloc[0][['model','macroF1']].to_dict(),
 },
}
with open(f'{RES}/summary_03.json','w') as f: json.dump(summary,f,indent=2,default=str)
print(json.dumps(summary, indent=2, default=str))
print('\nfiles written to ./results/:')
for p in sorted(os.listdir(RES)): print('  ', p)

### How to read these numbers

- If **MAE roughly triples** and **R² collapses** from V0 to V1, the standard random-split practice
  is substantially overstating real-world accuracy on this data. That is the paper's central claim,
  and it is now measured rather than asserted.
- A **negative R²** under V1 or V2 means the model does worse than simply predicting the mean —
  worth reporting plainly rather than hiding, because it shows how far some models are from
  transferring.
- If **F2/F3 beat F1 under V1 and V2** (not just under V0), material-physics features genuinely aid
  transfer. If they only help under V0, the model is memorising material identity instead.

**Next — Notebook 04:** the multi-task model (depth + width + defect mode with masked losses) and
uncertainty quantification (deep ensembles + conformal intervals + distance-to-training-support),
which together turn "the honest numbers are bad" into "here is how to deploy this responsibly".
